<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M11/M11_Lab1_Google_ADK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![Google ADK banner](https://raw.githubusercontent.com/mdehghani86/DADS5250-GenAI/main/labs/M11/assets/images/M11_Lab1_Google_ADK_banner.png)

# 🧭 Module 11 · Lab — Google ADK: Code-First Agents (Self-Reading)

**Difficulty:** ⭐⭐  ·  **Time:** ~12 min to read and run  ·  **Format:** self-reading — run each cell, watch what happens, answer the reflection prompts.

You have built agents with the OpenAI and Anthropic SDKs. Google's **Agent Development Kit (ADK)** takes a different stance: it is **code-first**. You write plain Python functions, hand them to an `Agent`, and ADK reads their type hints and docstrings to build the tool schema **for you** — no hand-written JSON. This short lab shows the ADK way and compares it to what you already know.


## 🔧 1. Setup

Install the ADK and the course utils. `nest_asyncio` lets ADK's async runner work inside a notebook. Run once per Colab runtime.


In [ ]:
# ==========================================================
# 1. Setup: install Google ADK + utils, enable async-in-notebook
# ==========================================================
%pip -q install google-adk dads5250==0.2.0 nest_asyncio

import os, asyncio, nest_asyncio         # async plumbing for notebooks
nest_asyncio.apply()                     # allow asyncio.run() inside Colab/Jupyter
from dads5250 import pp, pretty_print    # course utils


## ✅ 2. API check

ADK runs on **Gemini**, which has a **free tier** — get a key at [Google AI Studio](https://aistudio.google.com/app/apikey). The key is read from a Colab Secret named `GEMINI_API_KEY`, then an environment variable, then a hidden prompt. On JupyterHub set `GEMINI_API_KEY` via `export` or paste it at the prompt.


In [ ]:
# ==========================================================
# 2. API check: point ADK at Gemini (free tier, no Vertex)
# ==========================================================
from getpass import getpass

def _get_secret(name):
    try:                                          # 1) Colab Secret
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    if os.environ.get(name):                      # 2) environment variable
        return os.environ[name]
    return getpass(f"Enter {name}: ")             # 3) hidden prompt

ADK_MODEL = "gemini-2.5-flash"                     # free-tier Gemini model
# ADK reads GOOGLE_API_KEY; point it at the Gemini key and use AI Studio (not Vertex)
os.environ["GOOGLE_API_KEY"] = _get_secret("GEMINI_API_KEY")
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"

try:
    from google.adk.agents import Agent
    status = "ADK imported -- ready"
except Exception as e:
    status = f"ADK NOT ready -- {type(e).__name__}"

pp({"Google ADK": status, "model": ADK_MODEL}, title="API check")


## 🧠 3. Why ADK is different: tools are just Python

With OpenAI and Anthropic you write a JSON Schema for every tool by hand. ADK instead **reads your function** — its name, type hints, and docstring — and generates that schema automatically. So the tool *is* the function. Below are three ordinary functions; notice there is no JSON anywhere.

> **Read as you go:** the docstring is not just a comment here — ADK sends it to the model as the tool description, and the parameter type hints become the tool's input schema. Good docstrings = good tools.


In [ ]:
# ==========================================================
# 3. Tools = plain Python functions (ADK auto-builds the schema)
# ----------------------------------------------------------
# Purpose: show that a well-typed, well-documented function IS a tool.
# Defines: calculator(), web_search(), word_count() -- three simple tools
# ==========================================================
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, e.g. '675647 / 35'.

    Args:
        expression: A math expression using digits and + - * / ( ) only.
    """
    allowed = set("0123456789+-*/.() ")
    if not all(c in allowed for c in expression):
        return "Error: invalid characters"
    return str(round(eval(expression), 4))        # safe: only math chars allowed above

def web_search(query: str) -> str:
    """Look up a fact from a small built-in knowledge base.

    Args:
        query: What to look up, e.g. 'population of Boston'.
    """
    facts = {
        "population of boston": "Boston has a population of approximately 675,647 (2024).",
        "universities in boston": "Boston has approximately 35 colleges and universities.",
    }
    return facts.get(query.lower().strip(), "No result found for that query.")

def word_count(text: str) -> int:
    """Count the number of words in a piece of text.

    Args:
        text: The text whose words should be counted.
    """
    return len(text.split())


## 🤖 4. Build the agent and a run helper

An ADK `Agent` needs a name, a model, an instruction, and its tools — the same three functions from above, passed directly. ADK's runner is asynchronous, so we wrap it in a tiny `ask()` helper that runs one question and prints each tool call so you can *see* the agent reason step by step.


In [ ]:
# ==========================================================
# 4. The ADK research agent + an ask() helper
# ----------------------------------------------------------
# Purpose: define one agent that can chain the three tools, and a
#          helper that runs it and surfaces every tool call.
# Defines:
#   - research_agent : an ADK Agent with the three Python tools
#   - ask(prompt)    : run the agent once, print tool calls, return answer
# ==========================================================
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types

research_agent = Agent(
    name="research_agent",
    model=ADK_MODEL,
    instruction=("You are a careful research assistant. Use the tools to look up facts "
                 "and do math. Think step by step and combine tool results into a final answer."),
    tools=[calculator, web_search, word_count],   # <-- plain functions, no JSON schema
)

async def _run(prompt):
    runner = InMemoryRunner(agent=research_agent, app_name="adk_lab")
    session = await runner.session_service.create_session(app_name="adk_lab", user_id="student")
    answer = ""
    async for event in runner.run_async(
            user_id="student", session_id=session.id,
            new_message=types.Content(role="user", parts=[types.Part(text=prompt)])):
        if not (event.content and event.content.parts):
            continue
        for part in event.content.parts:
            if getattr(part, "function_call", None):     # the agent decided to use a tool
                fc = part.function_call
                print(f"  \U0001F4DE tool call: {fc.name}({dict(fc.args)})")
            if getattr(part, "text", None):
                answer = part.text                       # keep the latest text as the answer
    return answer

def ask(prompt):
    """Run the research agent on one prompt (sync wrapper for the notebook)."""
    print(f"\U0001F9D1 {prompt}")
    result = asyncio.run(_run(prompt))
    pretty_print(result, title="Agent answer")
    return result


## 🧪 5. Run it: a multi-step research question

This question needs **two tools in sequence** — a web lookup, then a division. Run the cell and watch the tool calls print in order before the final answer. That chaining, decided by the model, is the whole point of an agent.


In [ ]:
# ==========================================================
# 5. Watch the agent chain tools to answer one question
# ==========================================================
ask("What is the population of Boston divided by the number of universities there?")


**📝 Your observation** *(double-click to edit):* Which tools did the agent call, and in what order? Did it pick them itself, or did you tell it to? Write one or two sentences.

*Your answer here...*


## ⚖️ 6. ADK vs the SDKs you already know

Same agent idea, three different developer experiences. The row that matters most for ADK is **tool schema**.


In [ ]:
# ==========================================================
# 6. How the three agent SDKs compare
# ==========================================================
comparison = {
    "OpenAI (M06 / M12)": {
        "tool schema": "hand-written JSON Schema",
        "run model":  "you loop over tool_calls",
        "memory":     "manual (pass messages)",
        "free tier":  "no",
    },
    "Anthropic (M11 Agent SDK)": {
        "tool schema": "hand-written input_schema JSON",
        "run model":  "loop on stop_reason='tool_use'",
        "memory":     "manual (pass messages)",
        "free tier":  "no",
    },
    "Google ADK (this lab)": {
        "tool schema": "AUTO from function type hints + docstring",
        "run model":  "Runner handles the loop for you",
        "memory":     "Session service (built in)",
        "free tier":  "yes (Gemini)",
    },
}
pp(comparison, title="Agent SDK comparison")


## 🎯 7. Reflection & light exercise

1. **Observe:** In the comparison table, which two ADK features would save you the most time on a real project, and why?
2. **Read the code:** In `research_agent`, what exactly told the model that `web_search` takes a `query` string? (Hint: look at the function, not any JSON.)
3. **Try it (optional):** Add a fourth tool — a plain Python function `to_upper(text: str) -> str` with a clear docstring — to the agent's `tools` list, then ask a question that needs it. You wrote **no** schema.

## 📝 Summary

Google ADK is **code-first**: your typed, documented Python functions *are* the tools, a `Runner` drives the tool-calling loop, and a session service handles memory — all on free-tier Gemini. Same agent concepts you learned with OpenAI and Anthropic, with less boilerplate between you and a working agent.
